# v6: closed-loop beam tracking under image degradation + PAR/ResNet score fusion (h = 0, flight-disjoint)
**Accelerator:** GPU T4. **Inputs:** Scenario 23 + `resnet18-weights` (same as v5). Runtime about 1.5-2.5 h.

1. Closed-loop tracking (top-k probing, k = 1, 2, 3) on clean, blurred, low-light, occluded and 30% frame-dropped images.
2. Score fusion: PAR x ResNet-18 (log-linear, fixed weight 0.5, no tuning on test data), with the plain and the augmented ResNet.
3. Open-loop h = 0 accuracy of the same models under the same degradations.
4. Sampling period from the time stamps (fixed parser).

Output: download **results_v6.zip** from the Output tab.

In [ ]:
import os, glob, json, time, math, warnings, random, re, io
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.stats import wilcoxon, spearmanr
from sklearn.model_selection import KFold, GroupKFold
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

FAST = os.environ.get("ICICV_FAST", "0") == "1"   # local smoke test only
Q, W, D = 32, 8, 8
HORIZONS = [0, 1, 3, 5]
NFOLD = 3 if FAST else 5
GBT_ITERS = 30 if FAST else 150
CNN_EPOCHS = 1 if FAST else 6
CNN_AUG_EPOCHS = 1 if FAST else 8
GRU_EPOCHS = 2 if FAST else 60
IMG_RES = 64 if FAST else 160
ROB_H = [0, 3]
CL_K = [1, 3, 5]
SEED = 0
OUT = "/kaggle/working/out" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
REPORT = []
T_START = time.time()
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)
def save_report():
    open(f"{OUT}/report.txt", "w").write("\n".join(REPORT))
from concurrent.futures import ThreadPoolExecutor
def pmap(f, xs, workers=32):
    with ThreadPoolExecutor(workers) as ex: return list(ex.map(f, xs))

import torch, torchvision, shutil
torch.backends.cudnn.benchmark = True
NO_WEIGHTS = os.environ.get("ICICV_NOWEIGHTS", "0") == "1"
if not NO_WEIGHTS:
    _ckdir = os.path.join(torch.hub.get_dir(), "checkpoints"); os.makedirs(_ckdir, exist_ok=True)
    _w = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True) if "resnet18" in os.path.basename(p).lower()]
    if _w:
        shutil.copy(_w[0], os.path.join(_ckdir, "resnet18-f37072fd.pth")); log("ResNet-18 weights copied from", _w[0])
    try:
        torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT); log("ImageNet weights OK")
    except Exception as e:
        raise RuntimeError("ResNet-18 weights not found: Add Input -> resnet18-weights dataset") from e
def resnet18():
    return torchvision.models.resnet18(weights=None if NO_WEIGHTS else torchvision.models.ResNet18_Weights.DEFAULT)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
log("device:", DEV, "| FAST:", FAST)

## 1. Load Scenario 23

In [ ]:
cands = sorted(set(glob.glob("/kaggle/input/**/*.csv", recursive=True) + glob.glob("./**/scenario23*.csv", recursive=True)),
               key=lambda p: (("scenario23" in os.path.basename(p).lower()) * 10) - len(p) / 1000, reverse=True)
CSV = None
for p in cands:
    try:
        cols = pd.read_csv(p, nrows=2).columns.str.lower()
        if any("pwr" in c or "power" in c for c in cols): CSV = p; break
    except Exception: pass
assert CSV, "Scenario 23 CSV not found under /kaggle/input"
BASE = os.path.dirname(CSV)
df = pd.read_csv(CSV)
log("CSV:", CSV, df.shape); log("columns:", list(df.columns))

_cache = {}
def resolve(rel):
    rel = str(rel).strip()
    for c in [os.path.join(BASE, rel), os.path.join(BASE, rel.lstrip("./")), rel]:
        if os.path.exists(c): return c
    if not _cache:
        for f in glob.glob(os.path.join(BASE, "**", "*"), recursive=True): _cache.setdefault(os.path.basename(f), f)
    return _cache.get(os.path.basename(rel))
def load_val(v):
    if isinstance(v, str) and (v.endswith(".txt") or "/" in v):
        p = resolve(v); return None if p is None else np.loadtxt(p).astype(float).ravel()
    try: return np.array([float(v)])
    except Exception: return None
def find_col(*keys, exclude=()):
    for c in df.columns:
        lc = c.lower()
        if all(k in lc for k in keys) and not any(e in lc for e in exclude): return c
    return None

c_pwr = find_col("pwr") or find_col("power"); c_rgb = find_col("rgb") or find_col("image")
c_loc = find_col("unit2", "loc") or find_col("loc", exclude=("unit1",)); c_seq = find_col("seq")
c_bs = find_col("unit1", "loc"); c_idx = find_col("index"); c_time = find_col("time")
extra = {k: find_col("unit2", k) or find_col(k, exclude=("unit1",)) for k in ["height", "distance", "speed", "pitch", "roll"]}
extra = {k: c for k, c in extra.items() if c is not None and c != c_time}
log("power:", c_pwr, "rgb:", c_rgb, "loc:", c_loc, "seq:", c_seq, "time:", c_time, "extra:", extra)
if c_seq is not None and c_idx is not None:
    df = df.sort_values([c_seq, c_idx], kind="stable").reset_index(drop=True)

t0 = time.time(); P64 = np.stack(pmap(load_val, df[c_pwr].tolist()))
P = P64[:, ::2] if (P64.shape[1] == 64 and Q == 32) else P64
best = P.argmax(1)
loc = np.stack([x[:2] for x in pmap(load_val, df[c_loc].tolist())])
lat0, lon0 = loc[:, 0].mean(), loc[:, 1].mean()
if c_bs is not None:
    b0 = load_val(df[c_bs].iloc[0]); lat0, lon0 = b0[0], b0[1]
R_E = 6371000.0
Fd = {"east": np.deg2rad(loc[:, 1] - lon0) * R_E * np.cos(np.deg2rad(lat0)), "north": np.deg2rad(loc[:, 0] - lat0) * R_E}
for k, c in extra.items():
    Fd[k] = np.array([x[0] if x is not None else np.nan for x in pmap(load_val, df[c].tolist())])
F = pd.DataFrame(Fd)
F["az"] = np.degrees(np.arctan2(F["east"], F["north"]))
if "height" in F and "distance" in F: F["elev"] = np.degrees(np.arctan2(F["height"], F["distance"].clip(lower=1)))
F = F.fillna(F.median())
TEL_COLS = list(F.columns); TEL = F[TEL_COLS].values.astype(float)
seq = df[c_seq].values if c_seq else np.zeros(len(df), int)
_, seq = np.unique(seq, return_inverse=True)
img_paths = pmap(resolve, df[c_rgb].tolist())
N, NS = len(df), seq.max() + 1
lens = np.bincount(seq)
log(f"txt parsed in {time.time()-t0:.0f} s; N={N}, flights={NS}, length min/median/max {lens.min()}/{int(np.median(lens))}/{lens.max()}, images {sum(p is not None for p in img_paths)}")
ok = seq[1:] == seq[:-1]
log(f"P(beam changes between consecutive samples) = {(best[1:] != best[:-1])[ok].mean():.3f}")

# sampling period from time stamps (seconds)
DT = float("nan")
if c_time is not None:
    def to_sec(v):
        nums = re.findall(r"\d+(?:\.\d+)?", str(v))
        if len(nums) >= 4 and '.' not in nums[-1] and '.' not in nums[-2]:
            return 3600 * float(nums[-4]) + 60 * float(nums[-3]) + float(nums[-2]) + float(nums[-1]) / 10 ** len(nums[-1])
        if len(nums) >= 3:
            h_, m_, s_ = float(nums[-3]), float(nums[-2]), float(nums[-1]); return 3600 * h_ + 60 * m_ + s_
        return np.nan
    log('time stamp raw examples:', [repr(v) for v in df[c_time].iloc[:3]])
    try:
        _tt = pd.to_datetime(df[c_time].astype(str).str.strip("[]'\" "), errors='coerce')
        ts = (_tt - _tt.min()).dt.total_seconds().values if _tt.notna().mean() > 0.9 else np.array([to_sec(v) for v in df[c_time]])
    except Exception:
        ts = np.array([to_sec(v) for v in df[c_time]])
    d = np.diff(ts)[ok]; d = d[np.isfinite(d) & (d > 0) & (d < 10)]
    log(f'valid time differences: {len(d)}')
    if len(d):
        DT = float(np.median(d))
        log(f"time stamp example: {df[c_time].iloc[0]!r}; sampling period median {DT*1e3:.1f} ms (IQR {np.percentile(d,25)*1e3:.1f}-{np.percentile(d,75)*1e3:.1f} ms)")
save_report()

## 2. Image cues (causal and non-causal background subtraction) and ResNet-18 embedding

In [ ]:
import cv2
def read_small(p):
    im = cv2.imread(p, cv2.IMREAD_REDUCED_COLOR_2) if p else None
    if im is None: return np.zeros((270, 480, 3), np.uint8)
    if im.shape[:2] != (270, 480): im = cv2.resize(im, (480, 270), interpolation=cv2.INTER_AREA)
    return im
t0 = time.time(); IMG480 = np.stack(pmap(read_small, img_paths, 16))
log(f"images read in {time.time()-t0:.0f} s, {IMG480.shape}, {IMG480.nbytes/1e9:.1f} GB")

def _detect(dimg, prev, H, Wd):
    lab, n = ndimage.label(dimg > max(dimg.mean() + 5 * dimg.std(), 8.0))
    if n == 0: return None
    idx = np.arange(1, n + 1)
    area = np.asarray(ndimage.sum(np.ones_like(dimg), lab, idx)); mass = np.asarray(ndimage.sum(dimg, lab, idx))
    okk = np.where((area >= 2) & (area <= 0.003 * H * Wd))[0]
    if len(okk) == 0: return None
    com = np.array(ndimage.center_of_mass(dimg, lab, idx[okk]))
    cx, cy = com[:, 1] / Wd, com[:, 0] / H
    score = mass[okk] if prev is None else mass[okk] / (1 + 20 * np.hypot(cx - prev[0], cy - prev[1]))
    k = int(np.argmax(score)); return cx[k], cy[k], area[okk][k] / (H * Wd)

def localise(IMGC, causal=True):
    # causal=True: background = median of every 5th frame in the past 300 frames (only frames <= chunk start)
    # causal=False (v4): median of every 5th frame within +-150 frames (uses future frames)
    G = np.full((N, 4), np.nan, np.float32); G[:, 2:] = 0
    for s in range(NS):
        ids = np.where(seq == s)[0]
        gray = np.stack([cv2.cvtColor(IMGC[i], cv2.COLOR_BGR2GRAY) for i in ids]).astype(np.float32)
        L, H, Wd = gray.shape; prev = None
        for c0 in range(0, L, 25):
            if causal:
                bg = np.median(gray[max(0, c0 - 300):c0 + 1][::-1][::5], 0) if c0 >= 20 else None
            else:
                bg = np.median(gray[max(0, c0 - 150):min(L, c0 + 175):5], 0)
            for j in range(c0, min(L, c0 + 25)):
                b = bg
                if b is None:                       # flight start: only past frames 0..j (needs >= 5)
                    if j < 4: continue
                    b = np.median(gray[:j + 1], 0)
                r = _detect(cv2.GaussianBlur(np.abs(gray[j] - b), (5, 5), 0), prev, H, Wd)
                if r is not None:
                    G[ids[j]] = [r[0], r[1], r[2], 1.0]; prev = (r[0], r[1])
    for s in range(NS):                              # causal hold of missed detections
        last = None
        for i in np.where(seq == s)[0]:
            if G[i, 3] == 1: last = G[i, :3].copy()
            elif last is not None: G[i, :3] = last
    return np.nan_to_num(G, nan=0.5)

t0 = time.time(); G_causal = localise(IMG480, causal=True); CUE_MS = (time.time() - t0) / N * 1e3
G_noncausal = G_causal   # v6: non-causal cue not needed
for nm, G in [("causal", G_causal), ("non-causal (v4)", G_noncausal)]:
    v = G[:, 3] == 1
    log(f"cues {nm}: detection rate {v.mean():.3f}, Spearman(x-centroid, beam) = {spearmanr(G[v,0], best[v]).correlation:.3f}")
log(f"causal cue time {CUE_MS:.2f} ms/frame (single process, incl. per-flight median)")

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
def small_rgb(IMGC):
    return np.stack([cv2.cvtColor(cv2.resize(im, (IMG_RES, IMG_RES), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB) for im in IMGC])
def to_gpu_u8(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(DEV)       # (N,3,H,W) uint8
def norm(xu8): return ((xu8.float() / 255.) - MEAN) / STD

@torch.no_grad()
def embed(IMT):
    net = resnet18(); net.fc = torch.nn.Identity(); net.eval().to(DEV)
    out = []
    for i in range(0, IMT.shape[0], 256):
        with torch.autocast(DEV, enabled=DEV == "cuda"):
            out.append(net(norm(IMT[i:i+256])).float().cpu().numpy())
    return np.concatenate(out)
t0 = time.time(); IMT = {"clean": to_gpu_u8(small_rgb(IMG480))}; E_raw = {"clean": embed(IMT["clean"])}
log(f"small images on {DEV} + embedding in {time.time()-t0:.0f} s")
save_report()

## 3. Degraded test sets (computed once; models are trained on clean data only)

In [ ]:
def low_light(im, rng):
    x = (im / 255.) ** 2.5 * 0.5; x = x + rng.normal(0, 0.03, x.shape); return (np.clip(x, 0, 1) * 255).astype(np.uint8)
def blur(im, rng): return cv2.GaussianBlur(im, (0, 0), 4)
def occlusion(im, rng):
    H, Wd = im.shape[:2]; h, w = H // 2, Wd // 2; y0, x0 = rng.integers(0, H - h), rng.integers(0, Wd - w)
    im = im.copy(); im[y0:y0+h, x0:x0+w] = 0; return im
G_by = {"clean": G_causal}; KEEP = {"clean": np.arange(N)}
for dname, fn in [("low-light", low_light), ("blur", blur), ("occlusion", occlusion)]:
    t0 = time.time(); rng = np.random.default_rng(1)
    D480 = np.stack([fn(im, rng) for im in IMG480])
    G_by[dname] = localise(D480, causal=True); IMT[dname] = to_gpu_u8(small_rgb(D480)); E_raw[dname] = embed(IMT[dname]); KEEP[dname] = np.arange(N)
    del D480; log(f"degraded set {dname} ready in {time.time()-t0:.0f} s")
rng = np.random.default_rng(1); drop = rng.random(N) < 0.3; keep = np.arange(N)
for i in range(1, N):
    if drop[i] and seq[i] == seq[i - 1]: keep[i] = keep[i - 1]
G_by["frame-drop 30%"] = G_causal[keep]; E_raw["frame-drop 30%"] = E_raw["clean"][keep]; KEEP["frame-drop 30%"] = keep
IMKEY = {k: ("clean" if k == "frame-drop 30%" else k) for k in G_by}       # which image tensor + index map
DEGS = list(G_by.keys())
del IMG480; import gc; gc.collect()
save_report()

## 4. Features, tree models and metrics

In [ ]:
def build_index(h):
    T, TG = [], []
    for s in range(NS):
        ids = np.where(seq == s)[0]
        for j in range(W, len(ids) - h):
            T.append(ids[j]); TG.append(ids[j + h])
    return np.array(T), np.array(TG)

POS_COLS = [TEL_COLS.index(c) for c in ["east", "north", "height"] if c in TEL_COLS]
def feats(T, G, E, B=None):
    B = best if B is None else B                     # beam history source (true or closed-loop estimate)
    hist = np.stack([T - k for k in range(1, W + 1)], 1)
    bh = B[hist]; b_prev = B[T - 1]
    blk = {"tel": TEL[T], "pos": TEL[T][:, POS_COLS],
           "tel_d": np.concatenate([TEL[T] - TEL[T - k] for k in (1, 3, W - 1)], 1),
           "beam_abs": bh.astype(float), "beam_rel": (bh - b_prev[:, None]).astype(float),
           "b_prev": b_prev[:, None].astype(float), "img_geo": G[T],
           "img_geo_d": np.concatenate([G[T, :3] - G[T - k, :3] for k in (1, 3, W - 1)], 1)}
    if E is not None: blk["emb"] = E[T]
    return blk, b_prev

PAR_BLK = ["b_prev", "beam_rel", "tel", "tel_d", "img_geo", "img_geo_d"]
# name: (blocks, residual target?, cue variant, protocols)
FEATURE_SETS = {
    "GPS-KNN":          (["pos"], False, "c", ("random", "flight")),
    "GPS":              (["tel"], False, "c", ("random", "flight")),
    "GPS+hist":         (["tel", "tel_d", "beam_abs"], False, "c", ("random", "flight")),
    "Vision(emb)":      (["emb"], False, "c", ("random", "flight")),
    "Fusion(emb+GPS)":  (["emb", "tel"], False, "c", ("random", "flight")),
    "Fusion+hist":      (["emb", "tel", "tel_d", "beam_abs"], False, "c", ("random", "flight")),
    "PAR-noImg":        (["b_prev", "beam_rel", "tel", "tel_d"], True, "c", ("random", "flight")),
    "PAR (proposed)":   (PAR_BLK, True, "c", ("random", "flight")),
    # ablations (flight-disjoint only)
    "Abl: PAR features, absolute target": (PAR_BLK, False, "c", ("flight",)),
    "Abl: history-only residual":         (["b_prev", "beam_rel"], True, "c", ("flight",)),
    "Abl: PAR without telemetry":         (["b_prev", "beam_rel", "img_geo", "img_geo_d"], True, "c", ("flight",)),
    "Abl: PAR non-causal cues (v4)":      (PAR_BLK, True, "nc", ("flight",)),
}
ROB_TREE = ["Vision(emb)", "Fusion+hist", "PAR-noImg", "PAR (proposed)"]

import lightgbm as lgb
def gbt(seed, n_jobs=4):
    return lgb.LGBMClassifier(n_estimators=GBT_ITERS, learning_rate=0.1, num_leaves=31, min_child_samples=20,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                              random_state=seed, verbose=-1, n_jobs=n_jobs)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def knn(seed): return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, weights="distance"))

DCLS = np.arange(-D, D + 1)
def scores_to_beams(proba, classes, b_prev, residual):
    S = np.zeros((len(proba), Q))
    if residual:
        for j, dlt in enumerate(classes):
            np.add.at(S, (np.arange(len(proba)), np.clip(b_prev + dlt, 0, Q - 1)), proba[:, j])
    else:
        S[:, classes] = proba
    return S
PERS_ORDER = [(0, 1.0), (1, .5), (-1, .45), (2, .2), (-2, .15)]
def persistence_scores(b_prev):
    S = np.full((len(b_prev), Q), 1e-6)
    for d, w in PERS_ORDER:
        t = b_prev + d; m = (t >= 0) & (t < Q); S[np.where(m)[0], t[m]] = w
    return S
def metrics(S, y, tg):
    order = np.argsort(-S, 1)
    m = {f"top{k}": float(np.mean((order[:, :k] == y[:, None]).any(1))) for k in (1, 3, 5)}
    pl = 10 * np.log10(np.maximum(P[tg, y], 1e-12) / np.maximum(P[tg, order[:, 0]], 1e-12))
    m["pl_db"] = float(pl.mean()); m["pl95_db"] = float(np.percentile(pl, 95))
    return m, (order[:, 0] == y)

## 5. Neural baselines: ResNet-18 (plain, augmented, + history with residual target) and GRU sequence model

In [ ]:
class CNNHead(torch.nn.Module):
    def __init__(self, n_out, aux_dim=0):
        super().__init__()
        self.body = resnet18(); self.body.fc = torch.nn.Identity(); self.aux_dim = aux_dim
        self.head = torch.nn.Linear(512, n_out) if aux_dim == 0 else torch.nn.Sequential(
            torch.nn.Linear(512 + aux_dim, 256), torch.nn.ReLU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, n_out))
    def forward(self, x, aux=None):
        f = self.body(x)
        return self.head(f if self.aux_dim == 0 else torch.cat([f, aux], 1))

def augment(x, g):
    # x: normalised batch on GPU. Photometric + blur + cut-out augmentation (per sample)
    B = x.shape[0]
    x = x * STD + MEAN                                             # back to [0,1]
    gam = torch.exp(torch.empty(B, 1, 1, 1, device=x.device).uniform_(-0.7, 1.0, generator=g))
    bri = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0.4, 1.3, generator=g)
    x = (x.clamp(1e-4, 1) ** gam) * bri
    x = x + 0.03 * torch.randn(x.shape, device=x.device, generator=g) * (torch.rand(B, 1, 1, 1, device=x.device, generator=g) < 0.5)
    mb = torch.rand(B, device=x.device, generator=g) < 0.3
    if mb.any():
        x[mb] = torchvision.transforms.functional.gaussian_blur(x[mb], kernel_size=13, sigma=float(1 + 2 * torch.rand(1, generator=g, device=x.device)))
    H, Wd = x.shape[2:]
    for i in torch.where(torch.rand(B, device=x.device, generator=g) < 0.5)[0].tolist():
        h = int(H * (0.2 + 0.3 * torch.rand(1, generator=g, device=x.device))); w = int(Wd * (0.2 + 0.3 * torch.rand(1, generator=g, device=x.device)))
        y0 = int((H - h) * torch.rand(1, generator=g, device=x.device)); x0 = int((Wd - w) * torch.rand(1, generator=g, device=x.device))
        x[i, :, y0:y0+h, x0:x0+w] = 0
    return (x.clamp(0, 1) - MEAN) / STD

def train_cnn(T_tr, y_tr, n_out, seed, aux_tr=None, aug=False, epochs=CNN_EPOCHS):
    torch.manual_seed(seed); g = torch.Generator(device=DEV); g.manual_seed(seed)
    net = CNNHead(n_out, 0 if aux_tr is None else aux_tr.shape[1]).to(DEV).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda"); lossf = torch.nn.CrossEntropyLoss(); bs = 64
    yt = torch.from_numpy(y_tr).long().to(DEV); at = None if aux_tr is None else torch.from_numpy(aux_tr).float().to(DEV)
    Tt = torch.from_numpy(T_tr).long().to(DEV)
    for ep in range(epochs):
        net.train(); perm = torch.from_numpy(np.random.default_rng(seed + ep).permutation(len(T_tr))).to(DEV)
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]
            x = norm(IMT["clean"][Tt[b]]).contiguous(memory_format=torch.channels_last)
            if aug: x = augment(x, g)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                loss = lossf(net(x, None if at is None else at[b]), yt[b])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return net.eval()

@torch.no_grad()
def predict_cnn(net, T_te, imkey="clean", keep=None, aux_te=None):
    idx = T_te if keep is None else keep[T_te]
    It = torch.from_numpy(idx).long().to(DEV); out = []
    at = None if aux_te is None else torch.from_numpy(aux_te).float().to(DEV)
    for i in range(0, len(idx), 256):
        x = norm(IMT[imkey][It[i:i+256]]).contiguous(memory_format=torch.channels_last)
        with torch.autocast(DEV, enabled=DEV == "cuda"):
            o = net(x, None if at is None else at[i:i+256])
        out.append(torch.softmax(o.float(), 1).cpu().numpy())
    return np.concatenate(out)

def aux_matrix(blk, tr):
    A = np.concatenate([blk["b_prev"] / Q, blk["beam_rel"] / Q, blk["tel"], blk["tel_d"]], 1)
    mu, sd = A[tr].mean(0), A[tr].std(0) + 1e-6
    return ((A - mu) / sd).astype(np.float32)

class GRUNet(torch.nn.Module):
    def __init__(self, n_tel):
        super().__init__()
        self.emb = torch.nn.Embedding(Q, 16); self.gru = torch.nn.GRU(16 + n_tel, 64, batch_first=True)
        self.out = torch.nn.Linear(64, Q)
    def forward(self, b, tel):
        h, _ = self.gru(torch.cat([self.emb(b), tel], 2)); return self.out(h[:, -1])
def gru_inputs(T, B=None):
    B = best if B is None else B
    steps = np.stack([T - W + i for i in range(W)], 1)                # t-W .. t-1 (beams), telemetry at the next frame
    return B[steps], TEL[steps + 1]
def train_gru(T_tr, y_tr, seed):
    torch.manual_seed(seed)
    b, tel = gru_inputs(T_tr); mu, sd = tel.reshape(-1, tel.shape[2]).mean(0), tel.reshape(-1, tel.shape[2]).std(0) + 1e-6
    net = GRUNet(tel.shape[2]).to(DEV); opt = torch.optim.Adam(net.parameters(), 1e-3); lossf = torch.nn.CrossEntropyLoss()
    bt = torch.from_numpy(b).long().to(DEV); tt = torch.from_numpy((tel - mu) / sd).float().to(DEV); yt = torch.from_numpy(y_tr).long().to(DEV)
    for ep in range(GRU_EPOCHS):
        perm = torch.from_numpy(np.random.default_rng(seed + ep).permutation(len(T_tr))).to(DEV)
        for i in range(0, len(perm), 256):
            j = perm[i:i+256]; opt.zero_grad(); lossf(net(bt[j], tt[j]), yt[j]).backward(); opt.step()
    net.eval(); net.mu, net.sd = mu, sd; return net
@torch.no_grad()
def predict_gru(net, T):
    b, tel = gru_inputs(T)
    return torch.softmax(net(torch.from_numpy(b).long().to(DEV), torch.from_numpy((tel - net.mu) / net.sd).float().to(DEV)), 1).cpu().numpy()

## 6. Closed-loop beam tracking (h = 0): only the top-k predicted beams are measured

In [ ]:
def closed_loop(T_te, predict_scores, k):
    # Frames 0..W-1 of each test flight: full sweep (history known). From frame W on, the history is the
    # closed-loop estimate; k beams are measured and the strongest measured beam becomes the new estimate.
    est = best.copy()                                      # only test-flight entries are overwritten below
    flights = np.unique(seq[T_te]); ids_by = {s: np.where(seq == s)[0] for s in flights}
    Lmax = max(len(v) for v in ids_by.values())
    hit, loss, flt = [], [], []
    for j in range(W, Lmax):
        T = np.array([ids_by[s][j] for s in flights if len(ids_by[s]) > j])
        if len(T) == 0: continue
        S = predict_scores(T, est)
        probe = np.argsort(-S, 1)[:, :k]
        pick = probe[np.arange(len(T)), np.argmax(P[T[:, None], probe], 1)]
        est[T] = pick
        hit.append(pick == best[T]); loss.append(10 * np.log10(np.maximum(P[T, best[T]], 1e-12) / np.maximum(P[T, pick], 1e-12))); flt.append(seq[T])
    hit, loss, flt = np.concatenate(hit), np.concatenate(loss), np.concatenate(flt)
    n_tot = sum(len(v) for v in ids_by.values())
    overhead = (sum(min(len(v), W) for v in ids_by.values()) * Q + len(hit) * k) / (n_tot * Q)
    return {"acc": float(hit.mean()), "pl_db": float(loss.mean()), "pl95_db": float(np.percentile(loss, 95)),
            "outage_3db": float((loss > 3).mean()), "overhead": float(overhead)}, hit, flt

## 7. v6 experiment: h = 0, flight-disjoint, open and closed loop, clean and degraded

In [ ]:
CL_K = [1, 2, 3]
h = 0
T, TG = build_index(h); y = best[TG]; groups = seq[T]; dlt = y - best[T - 1]; dclip = np.clip(dlt, -D, D)
log(f"h=0: {len(T)} samples, {len(np.unique(groups))} flights")
def fuse(S1, S2, a=0.5):
    L = a * np.log(S1 + 1e-6) + (1 - a) * np.log(S2 + 1e-6); L -= L.max(1, keepdims=True); E = np.exp(L)
    return E / E.sum(1, keepdims=True)
def aux_stats(blk, tr):
    A = np.concatenate([blk["b_prev"] / Q, blk["beam_rel"] / Q, blk["tel"], blk["tel_d"]], 1)
    return A[tr].mean(0), A[tr].std(0) + 1e-6
def aux_from(blk, mu, sd):
    A = np.concatenate([blk["b_prev"] / Q, blk["beam_rel"] / Q, blk["tel"], blk["tel_d"]], 1)
    return ((A - mu) / sd).astype(np.float32)

ol_rows, cl_rows, cl_pf = [], [], []
for fold, (tr, te) in enumerate(GroupKFold(NFOLD).split(T, groups=groups)):
    t_fold = time.time()
    blk, b_prev = feats(T, G_causal, None)
    trees = {}
    for name in ["PAR (proposed)", "PAR-noImg", "GPS+hist"]:
        blocks, residual, _, _ = FEATURE_SETS[name]
        trees[name] = (gbt(SEED + fold).fit(np.concatenate([blk[b] for b in blocks], 1)[tr], (dclip if residual else y)[tr]), blocks, residual)
    gru = train_gru(T[tr], y[tr], SEED + fold)
    net_r = train_cnn(T[tr], y[tr], Q, SEED + fold)
    net_a = train_cnn(T[tr], y[tr], Q, SEED + fold, aug=True, epochs=CNN_AUG_EPOCHS)
    mu, sd = aux_stats(blk, tr)
    net_h = train_cnn(T[tr], dclip[tr] + D, 2 * D + 1, SEED + fold, aux_tr=aux_from(blk, mu, sd)[tr])
    log(f"fold {fold}: models trained in {time.time()-t_fold:.0f} s")

    for dg in DEGS:
        Gd = G_by[dg]; kp = KEEP[dg] if dg == "frame-drop 30%" else None
        # memoryless camera scores: computed once for all test samples
        SC = {}
        for nm, net in [("ResNet-18", net_r), ("ResNet-18 + aug", net_a)]:
            A = np.zeros((N, Q)); A[T[te]] = predict_cnn(net, T[te], imkey=IMKEY[dg], keep=kp); SC[nm] = A
        def tree_sc(name):
            clf, blocks, residual = trees[name]
            def f(Tq, B):
                bl, bp = feats(Tq, Gd, None, B)
                return scores_to_beams(clf.predict_proba(np.concatenate([bl[b] for b in blocks], 1)), clf.classes_, bp, residual)
            return f
        par = tree_sc("PAR (proposed)")
        def gru_sc(Tq, B):
            b, tel = gru_inputs(Tq, B)
            with torch.no_grad():
                return torch.softmax(gru(torch.from_numpy(b).long().to(DEV), torch.from_numpy((tel - gru.mu) / gru.sd).float().to(DEV)), 1).cpu().numpy()
        def resh_sc(Tq, B):
            bl, bp = feats(Tq, Gd, None, B)
            return scores_to_beams(predict_cnn(net_h, Tq, imkey=IMKEY[dg], keep=kp, aux_te=aux_from(bl, mu, sd)), DCLS, bp, True)
        M = {"Persistence (local sweep)": lambda Tq, B: persistence_scores(B[Tq - 1]),
             "PAR (proposed)": par, "PAR-noImg": tree_sc("PAR-noImg"), "GPS+hist": tree_sc("GPS+hist"),
             "GRU (beam+telemetry seq.)": gru_sc,
             "ResNet-18": lambda Tq, B: SC["ResNet-18"][Tq], "ResNet-18 + aug": lambda Tq, B: SC["ResNet-18 + aug"][Tq],
             "ResNet-18 + history (residual)": resh_sc,
             "Fusion PAR x ResNet-18": lambda Tq, B: fuse(par(Tq, B), SC["ResNet-18"][Tq]),
             "Fusion PAR x ResNet-18 + aug": lambda Tq, B: fuse(par(Tq, B), SC["ResNet-18 + aug"][Tq])}
        for name, f in M.items():                       # open loop (oracle history)
            m, _ = metrics(f(T[te], best), y[te], TG[te])
            ol_rows.append({"degradation": dg, "fold": fold, "model": name, **m})
        for k in CL_K:                                  # closed loop
            for name, f in M.items():
                m, hit, flt = closed_loop(T[te], f, k)
                cl_rows.append({"degradation": dg, "k": k, "fold": fold, "model": name, **m})
                for fl in np.unique(flt):
                    mk = flt == fl; cl_pf.append({"degradation": dg, "k": k, "model": name, "flight": int(fl), "n": int(mk.sum()), "acc": float(hit[mk].mean())})
        log(f"fold {fold} {dg}: closed-loop k=2 " + ", ".join(f"{r['model']}={r['acc']:.3f}" for r in cl_rows if r["fold"] == fold and r["k"] == 2 and r["degradation"] == dg))
        pd.DataFrame(ol_rows).to_csv(f"{OUT}/open_loop_h0.csv", index=False); pd.DataFrame(cl_rows).to_csv(f"{OUT}/closed_loop.csv", index=False)
        pd.DataFrame(cl_pf).to_csv(f"{OUT}/closed_loop_per_flight.csv", index=False); save_report()
    log(f"fold {fold} done in {time.time()-t_fold:.0f} s (total {time.time()-T_START:.0f} s)")
    del net_r, net_a, net_h; torch.cuda.empty_cache() if DEV == "cuda" else None
OL, CL, CLPF = map(pd.DataFrame, (ol_rows, cl_rows, cl_pf))

## 8. Summary tables and paired tests

In [ ]:
log("\nOpen loop h=0 top-1 (oracle history), mean over folds:\n" + OL.groupby(["model", "degradation"])["top1"].mean().unstack().round(3).to_string())
for k in CL_K:
    log(f"\nClosed loop k={k} accuracy:\n" + CL[CL.k == k].groupby(["model", "degradation"])["acc"].mean().unstack().round(3).to_string())
    log(f"Closed loop k={k} mean power loss (dB):\n" + CL[CL.k == k].groupby(["model", "degradation"])["pl_db"].mean().unstack().round(3).to_string())
    log(f"Closed loop k={k} 3-dB outage:\n" + CL[CL.k == k].groupby(["model", "degradation"])["outage_3db"].mean().unstack().round(3).to_string())
log("\noverhead by k: " + CL.groupby("k")["overhead"].mean().round(3).to_string())

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); idx = np.where(np.isfinite(p))[0]
    order = idx[np.argsort(p[idx])]; m = len(order); run = 0
    for r, i in enumerate(order): run = max(run, min(1, (m - r) * p[i])); out[i] = run
    return out
tests = []
for ref in ["Fusion PAR x ResNet-18 + aug", "PAR (proposed)"]:
    for (dg, k), g in CLPF.groupby(["degradation", "k"]):
        a = g[g.model == ref].set_index("flight")["acc"]
        for m in g.model.unique():
            if m == ref: continue
            b = g[g.model == m].set_index("flight")["acc"]; j = a.index.intersection(b.index); d = a[j] - b[j]
            p = wilcoxon(d).pvalue if (d != 0).sum() > 5 else np.nan
            tests.append({"ref": ref, "degradation": dg, "k": k, "vs": m, "mean_diff_pp": 100 * d.mean(), "wins": int((d > 0).sum()), "losses": int((d < 0).sum()), "p": p})
TT = pd.DataFrame(tests); TT["p_holm"] = np.nan
for ref in TT.ref.unique():
    mk = TT.ref == ref; TT.loc[mk, "p_holm"] = holm(TT.loc[mk, "p"].values)
TT.to_csv(f"{OUT}/closed_loop_tests.csv", index=False)
log("\nClosed-loop paired tests, k=2 and k=3:\n" + TT[TT.k.isin([2, 3])].round(4).to_string())
summary = {"sampling_period_s": DT, "runtime_s": time.time() - T_START, "N": int(N), "folds": NFOLD}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1); save_report()
shutil.make_archive("/kaggle/working/results_v6" if os.path.isdir("/kaggle/working") else "./results_v6", "zip", OUT)
print("DONE in", round((time.time() - T_START) / 60), "min. Download results_v6.zip from the Output tab.")